In [1]:
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn import model_selection
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sb

In [2]:
# https://drive.google.com/file/d/1Rkq42BS8xWVIvIFzFW83wlWA3nrlRk6f/view?usp=sharing
file_id = '1Rkq42BS8xWVIvIFzFW83wlWA3nrlRk6f'
download_url = f'https://drive.google.com/uc?id={file_id}'
df = pd.read_csv(download_url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
# -------------------------
# Generar valores nulos y filas duplicadas
# -------------------------

rng = np.random.default_rng(42)
df_sucio = df.copy(deep=True)

# Convertir TotalCharges a numérico:
# los espacios en blanco existentes se convierten en nulos
df_sucio["TotalCharges"] = pd.to_numeric(
    df_sucio["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

# 1. Introducir nulos en el 5 % de los valores
# originalmente disponibles de cada columna seleccionada
columnas = ["tenure", "MonthlyCharges", "TotalCharges"]

for columna in columnas:
    indices_validos = df_sucio.index[
        df_sucio[columna].notna()
    ].to_numpy()

    cantidad = int(len(indices_validos) * 0.05)

    indices = rng.choice(
        indices_validos,
        size=cantidad,
        replace=False
    )

    df_sucio.loc[indices, columna] = np.nan

# 2. Agregar un 2 % de filas duplicadas
cantidad_duplicados = int(len(df_sucio) * 0.02)

indices = rng.choice(
    df_sucio.index.to_numpy(),
    size=cantidad_duplicados,
    replace=False
)

df_sucio = pd.concat(
    [df_sucio, df_sucio.loc[indices].copy()],
    ignore_index=True
)

# 3. Mostrar los resultados
print("Filas originales:", len(df))
print("Filas después de agregar duplicados:", len(df_sucio))
print("Filas duplicadas:", df_sucio.duplicated().sum())

print("\nValores nulos por columna:")
display(df_sucio.isna().sum().to_frame("Valores nulos"))

display(df_sucio.head())
# -------------------------
# Guardar el dataset sucio junto al cuaderno de Colab
# -------------------------

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# Autorizar el acceso a Google Drive
auth.authenticate_user()
servicio_drive = build("drive", "v3")

# ID de tu cuaderno de Colab
colab_id = "1XvwNXWW5_AGOKTid3WT7Dfc16hh7h-vz"

# Obtener la carpeta donde está guardado
cuaderno = servicio_drive.files().get(
    fileId=colab_id,
    fields="parents"
).execute()

carpetas = cuaderno.get("parents", [])

if not carpetas:
    raise ValueError("No se pudo identificar la carpeta del cuaderno.")

# Crear el CSV temporalmente en Colab
ruta_csv = "/content/Telco_Customer_Churn_Sucio.csv"
df_sucio.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

# Subirlo a la misma carpeta del cuaderno
archivo = servicio_drive.files().create(
    body={
        "name": "Telco_Customer_Churn_Sucio.csv",
        "parents": [carpetas[0]]
    },
    media_body=MediaFileUpload(ruta_csv, mimetype="text/csv"),
    fields="id, webViewLink"
).execute()

print("Archivo guardado en Google Drive:")
print(archivo["webViewLink"])

Filas originales: 7043
Filas después de agregar duplicados: 7183
Filas duplicadas: 140

Valores nulos por columna:


,Valores nulos
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,359
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Archivo guardado en Google Drive:
https://drive.google.com/file/d/1o1Ilpxk69avNJ0rP6DAooWObG6fE-EZ1/view?usp=drivesdk
